In [3]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Sequence
import operator
import time

# Core LangGraph components
from langgraph.graph import StateGraph, START, END # Import END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver # Import the saver

# LLM and messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

# Load API keys and set up tracing
load_dotenv()
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Intro to LangGraph"

# Define the State
class GraphState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    needs_approval: bool # Flag to track if approval is needed

# Define Nodes
def call_llm(state: GraphState):
    print("--- Calling LLM ---")
    llm = ChatOpenAI(model="gpt-4o")
    response = llm.invoke(state['messages'])
    
    # Check if the AI's response might need approval
    needs_approval_flag = "action required" in response.content.lower()
    print(f"AI Response: {response.content}")
    print(f"Needs Approval Flag: {needs_approval_flag}")
    
    # Update state
    return {"messages": [response], "needs_approval": needs_approval_flag}

def human_approval(state: GraphState):
    # Node called only if approval is needed
    print("\n--- Awaiting Human Approval ---")
    last_message = state['messages'][-1]
    print(f"AI Response to Review: {last_message.content}")
    # Graph will pause AFTER this node
    return {}

# Define Router for Dynamic Breakpoint
def check_for_approval(state: GraphState):
    # Router decides the next step based on the flag
    if state.get("needs_approval"):
        print("Routing to human approval.")
        return "request_approval"
    else:
        print("No approval needed. Ending.")
        return END # Signal to end the graph

# Build the Graph
workflow = StateGraph(GraphState)
workflow.add_node("llm", call_llm)
workflow.add_node("approve", human_approval)

workflow.set_entry_point("llm")

# Conditional edge after LLM calls the router
workflow.add_conditional_edges(
    "llm",
    check_for_approval,
    {
        "request_approval": "approve",
        END: END # Explicitly map the END condition
    }
)
workflow.add_edge("approve", END)

# Use SqliteSaver as a context manager
with SqliteSaver.from_conn_string(":memory:") as memory:
    # Compile the graph WITH interrupts and the memory saver
    app = workflow.compile(
        checkpointer=memory,
        interrupt_after=["approve"]
    )

    # --- Run 1: Approval NOT Needed ---
    config1 = {"configurable": {"thread_id": "dynamic-bp-thread-1"}}
    print("--- Test Run 1 (No Approval Expected) ---")
    # This invoke runs inside the 'with' block
    final_state_1 = app.invoke({"messages": [HumanMessage(content="Write a haiku about clouds.")]}, config1)
    print("\n--- Run 1 Finished ---")

    # --- Run 2: Approval Needed ---
    config2 = {"configurable": {"thread_id": "dynamic-bp-thread-2"}}
    print("\n\n--- Test Run 2 (Approval Expected) ---")
    # This input should trigger the needs_approval flag
    # This invoke also runs inside the 'with' block
    paused_state_2 = app.invoke({"messages": [HumanMessage(content="Confirm: action required for task XYZ.")]}, config2)

    # Graph pauses after 'approve' node
    print("\n--- Run 2 Paused ---")
    print("Paused State:")
    for msg in paused_state_2['messages']:
        print(f"- {msg.type}: {msg.content}")

    # Simulate approval and resume
    print("\n--- Resuming Run 2 ---")
    # This invoke also runs inside the 'with' block
    final_state_2 = app.invoke(None, config2) # Resume
    print("\n--- Run 2 Finished ---")

--- Test Run 1 (No Approval Expected) ---
--- Calling LLM ---
AI Response: Whispers in the sky,  
Dancing across the sunlight,  
Dreams in soft embrace.
Needs Approval Flag: False
No approval needed. Ending.

--- Run 1 Finished ---


--- Test Run 2 (Approval Expected) ---
--- Calling LLM ---
AI Response: Could you please provide more details about the task XYZ or what action you need to be confirmed? This will help me assist you better.
Needs Approval Flag: False
No approval needed. Ending.

--- Run 2 Paused ---
Paused State:
- human: Confirm: action required for task XYZ.
- ai: Could you please provide more details about the task XYZ or what action you need to be confirmed? This will help me assist you better.

--- Resuming Run 2 ---

--- Run 2 Finished ---
